# Are the animals categorical in shape space? (IBL autism dataset)

Each animal is a point -- its whole population of neurons, compared with the Procrustes
distance -- and the question is whether those points fall into discrete groups or form one
continuous cloud.

Silhouette cannot answer that directly: it needs at least two clusters, so it can never score
the one-cluster hypothesis. That hypothesis is simulated instead, by drawing points from a
**single Gaussian** matched to the real cloud's mean and covariance -- the real spread, none of
the lumpiness. Sweep k on the data, sweep k identically on each draw, and compare the best
silhouette with the null's own best. Because the null makes the same free choice of k,
choosing k by `argmax` costs nothing.

The same test on three sets of animals:

| panel | animals |
|---|---|
| **a** | all 36 |
| **b** | 26 mutants (Fmr1, Cntnap2, Shank3) |
| **c** | 10 controls (wild type) |

Each panel re-embeds its own submatrix and gets a Gaussian matched to it, so each is compared
with a null of its own size -- necessary, because silhouette rises as the number of points
falls. Only the z-scores are comparable across panels; the raw silhouettes are not.

Same test as `duszkiewicz_analyses/notebooks/hd_clustering.ipynb` panel d (head-direction
subjects) and `Posani/notebook.ipynb` section 4 (cortical regions).

In [ ]:
from pathlib import Path

from shapemetrics import paths

paths.set_figure("Figure4")
import numpy as np
import matplotlib.pyplot as plt
import tqdm

import shapemetrics as sm                                            # noqa: E402

OUT = paths.results("results_asd")
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"svg.fonttype": "none", "text.usetex": False})

# house style of the head-direction figure: grey is always the null, dark red the data
# The old house style (grey null, dark red data, 2.1in panels), which is what
# this notebook was drawn with. The package now carries both palettes under
# distinct names -- sm.PANEL/NULLC/OBS are the newer, richer one.
PANEL, NULLC, DATAC = sm.PANEL_SIMPLE, sm.NULL_GREY, sm.DATA_RED
N_NULL, N_PCS = 500, 20    # 500 draws: p resolves to 1/501, at a few minutes per null
MIN_NEURONS = 100          # per animal, so an animal is a population rather than a handful
GENO_NAME = {"N": "wild type", "F": "Fmr1", "C": "Cntnap2", "S": "Shank3"}
MUTANTS = "FCS"


## The data

One row per neuron -- 9 signed contrasts x 106 time bins of stimulus-evoked encoding kernel,
from the released GAM fits (`extract_asd_kernels.py`). Animals contributing fewer than
`MIN_NEURONS` neurons are dropped: each animal has to stand as a population.

In [ ]:
z = np.load(paths.derived("noel2025", "asd_kernels.npz"), allow_pickle=True)
X, animal, genotype = z["X"].astype(float), z["animal"], z["genotype"]

counts = {a: int((animal == a).sum()) for a in np.unique(animal)}
keep = np.array([counts[a] >= MIN_NEURONS for a in animal])
X, animal, genotype = X[keep], animal[keep], genotype[keep]

subjects = np.unique(animal)
geno_of = np.array([genotype[animal == s][0] for s in subjects])   # genotype per animal
is_mut = np.isin(geno_of, list(MUTANTS))
print(f"{len(X)} neurons, {len(subjects)} animals "
      f"({int((~keep).sum())} neurons dropped with their animals)")
for g in "NFCS":
    print(f"  {GENO_NAME[g]:<11} ({g})  {int((geno_of == g).sum()):>2} animals, "
          f"{int((genotype == g).sum()):>5} neurons")

## Subject space

Each animal's neurons become one point cloud -- its 954 features by `N_PCS` principal
components -- and animals are compared with the Procrustes distance. The matrix is cached, so
it is built once.

In [ ]:
f = OUT / "subject_procrustes.npz"
if f.exists():
    D_all = np.load(f)["D"]
else:
    D_all = sm.distance_matrix(X, animal, n_pcs=N_PCS)
    np.savez(f, D=D_all)

print(f"{D_all.shape[0]} x {D_all.shape[0]} Procrustes distances, "
      f"{D_all[np.triu_indices_from(D_all, 1)].mean():.3f} on average")


## The test

`embed` puts a distance matrix in a 5-dimensional Euclidean space, because k-means needs
coordinates. `silhouette_sweep` is Posani et al.'s statistic -- best-of-50-restarts k-means at
each k -- and `gaussian_null` repeats it on draws from one Gaussian matched to that embedding.
k runs from 2 to n - 1, and the null is given the same range.

`run` does that for one set of animals and caches it, so the three runs below differ only in
which rows and columns of the distance matrix they take. Each null is `N_NULL` sweeps, and a
sweep is 34 values of k with 50 restarts each, so this is the slow part of the notebook: a few
minutes per panel.

In [ ]:
def run(D, save_id):
    '''Data sweep and matched-Gaussian null for one set of animals, cached.

    The cache is keyed on `N_NULL` as well as on the file name, so changing the
    number of draws recomputes rather than silently reusing a null of a
    different size.
    '''
    ks = np.arange(2, len(D))                     # 2 .. n-1

    def compute():
        E = sm.mds(D)
        return dict(obs_sweep=sm.silhouette_sweep(E, ks), ks=ks, emb=E,
                    null_sweep=sm.gaussian_null(E, ks, N_NULL,
                                                progress=lambda n: tqdm.trange(n, desc=save_id)))

    d = sm.cached_npz(OUT / f"subject_gaussian_{save_id}.npz", compute,
                      valid=lambda d: len(d["null_sweep"]) == N_NULL)
    sweep, null_sweep, ks = d["obs_sweep"], d["null_sweep"], d["ks"]

    r = sm.null_stats(sweep.max(), null_sweep.max(1))   # the null picks its own best k too
    r.update(k=int(ks[sweep.argmax()]), n=len(D),
             k2=float(np.mean(ks[null_sweep.argmax(1)] == 2)))
    print(f"{save_id:<8} n = {r['n']:>2}   silhouette {r['obs']:.4f} (k = {r['k']})   "
          f"Gaussian null {r['null'].mean():.4f} +/- {r['null'].std():.4f}   "
          f"z = {r['z']:+.2f}, p = {r['p']:.4f}   (null picks k = 2 in {r['k2']:.0%})")
    return r


is_mut = np.isin(geno_of, list(MUTANTS))
res_all = run(D_all, "all")
res_mut = run(D_all[np.ix_(is_mut, is_mut)], "mutants")
res_ctl = run(D_all[np.ix_(~is_mut, ~is_mut)], "controls")


## The figure

Grey is always the null, dark red always the data.

In [ ]:
PANELS = [("all", res_all, "all genotypes"), ("mutants", res_mut, "mutants only"),
          ("controls", res_ctl, "controls only")]

fig, axes = plt.subplots(1, 3, figsize=(3 * PANEL, PANEL))
for ax, (_, r, title) in zip(axes, PANELS):
    sm.simple_null_panel(ax, r["obs"], r["null"], "best silhouette over k",
                  "one continuous cloud", "real animals",
                  title=f"{title} (n = {r['n']})", z=r["z"], p=r["p"])

sm.save_stem(fig, OUT / "asd_subject_gaussian")
plt.show()

print(f"{N_NULL} draws per null, so the smallest attainable p is {1 / (N_NULL + 1):.4f}\n")
for name, r, title in PANELS:
    print(f"{title:<13} n = {r['n']:>2}   {r['obs']:.4f} vs {r['null'].mean():.4f} "
          f"+/- {r['null'].std():.4f}   z = {r['z']:+.2f}, p = {r['p']:.4f}, k = {r['k']}")


## What it says

| panel | animals | n | silhouette | Gaussian null | z | p |
|---|---|---|---|---|---|---|
| a | all genotypes | 36 | 0.403 | 0.254 +/- 0.031 | **+4.78** | 0.0020 |
| b | mutants only | 26 | 0.285 | 0.255 +/- 0.035 | +0.86 | 0.16 |
| c | controls only | 10 | 0.552 | 0.388 +/- 0.077 | +2.12 | 0.032 |

The full set of animals is lumpy -- p = 0.0020 is the floor for 500 draws, so no draw from a
continuous cloud reached the data. Neither subset reproduces that on its own: the mutants are
indistinguishable from one continuous cloud, and the controls land at z = +2.12, which is
suggestive but rests on 10 animals, where a single outlier moves the silhouette a long way
(the null's own spread, 0.077, is more than twice the 36-animal one).

Read the z-scores, not the silhouettes. Silhouette rises as points get fewer -- the null's mean
climbs from 0.25 at n = 36 to 0.39 at n = 10 -- so the raw 0.552 of the controls is not
"better clustered" than the 0.403 of the full set.

Note also that all three win at k = 2, and that k-means will halve any elongated cloud: the
null itself picks k = 2 in 61%, 37% and 84% of its own draws. k = 2 winning in the data carries
little information on its own.

For contrast, the same test is negative in the two other datasets it has been run on: the
head-direction subjects (`hd_clustering.ipynb` panel d, z = +0.58) and the cortical regions of
`Posani/notebook.ipynb` (z = +0.47) are both organised but continuous.